# Pretrained WorM VSR diagnostic (Appendix Q)

Current-paper experiment. The official setup source notebook is included next to this file as `worm_vsr_setup.ipynb`; result CSVs are included in `results/worm_vsr/`. The human reference remains sealed and is not part of this release.

The scientific logic is unchanged; only file discovery was made repo-local with the original Drive paths retained as fallbacks.


In [ ]:
# 1) Mount Drive, point at the real project folder, inventory what survived
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

import os, math, copy, random, json, shutil, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader, Subset

_cwd = Path.cwd().resolve()
_repo = next((p for p in [_cwd, *_cwd.parents] if (p / "results" / "worm_vsr").exists()), _cwd)
_local_results = _repo / "results" / "worm_vsr"
DRIVE_ROOT = Path(os.environ.get("WORM_RESULT_DIR", str(_local_results if _local_results.exists() else Path(os.environ.get("RESULTS_DIR", "results/ICLR2027_WorM_L3_handover")))))
RESULT_DIR = DRIVE_ROOT                      # csvs live directly here
CKPT_DIR   = DRIVE_ROOT / "checkpoints"      # new: model state persists now
CACHE_DIR  = DRIVE_ROOT / "cache"            # new: feature cache
HUMAN_DIR  = Path(os.environ.get("WORM_HUMAN_DIR", os.environ.get("RESULTS_DIR", "results/ICLR2027_human_wm")))
NB_DIR     = Path(os.environ.get("WORM_SETUP_DIR", str(_repo / "experiments" / "appendix_q_worm_vsr")))

for d in (CKPT_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)
assert DRIVE_ROOT.is_dir(), f"not found: {DRIVE_ROOT}"
print("RESULT_DIR:", RESULT_DIR)

SAVED = {}
for f in sorted(RESULT_DIR.glob("*.csv")):
    try:
        SAVED[f.stem] = pd.read_csv(f)
        print(f"  loaded {f.name}  {SAVED[f.stem].shape}")
    except Exception as e:
        print(f"  FAILED {f.name}: {e}")

EXPECTED = ["recurrent_viability_high_n", "two_sided_screen_high_n",
            "two_sided_screen_repaired_data", "retrieval_stage2_seeds",
            "retrieval_ladder", "representation_probe_ceiling",
            "commonmode_decomposition"]
print("\nmissing:", [k for k in EXPECTED if k not in SAVED] or "none")

RECURRENT_HIGH_N = SAVED.get("recurrent_viability_high_n")
POOLED_BIG       = SAVED.get("two_sided_screen_repaired_data")
LADDER_DF        = SAVED.get("retrieval_ladder")
PROBE            = SAVED.get("representation_probe_ceiling")

for name, df in [("recurrent viability (high n)", RECURRENT_HIGH_N),
                 ("two-sided screen (repaired data)", POOLED_BIG)]:
    if df is not None:
        print("\n" + name); display(df.round(4))

print(f"\nhuman reference present: {(HUMAN_DIR/'worm_vsr_human_reference.csv').exists()}"
      "  (NOT loaded -- stays sealed)")
print("stage2 checkpoints here:",
      sorted(p.name for p in CKPT_DIR.glob('*.pt')) or "none yet")

In [ ]:
# 2) Find the FIXED notebook and show its setup cells (dry run -- nothing executes)
NB_CANDIDATES = [
    NB_DIR / "worm_vsr_setup.ipynb",
    NB_DIR / "worm_vsr_l3_pretrained_hybrid_handover_FIXED.ipynb",
    NB_DIR / "worm_vsr_l3_pretrained_hybrid_handover_colab.ipynb",
    NB_DIR / "exact_worm_vsr_causal_handover_colab.ipynb",
]
SRC_NB = next((p for p in NB_CANDIDATES if p.exists()), None)
assert SRC_NB is not None, "none of these exist:\n" + "\n".join(map(str, NB_CANDIDATES))
print("source notebook:", SRC_NB.name, f"({SRC_NB.stat().st_size/1e6:.2f} MB)\n")

_nb = json.loads(SRC_NB.read_text())
CODE_CELLS = [(i, "".join(c["source"]))
              for i, c in enumerate(_nb["cells"]) if c["cell_type"] == "code"]

PROBE_MARKERS = ("7b) Probe", "official_hidden_states", "HIDDEN_SIZE =")
SETUP_END = next((n for n, (i, s) in enumerate(CODE_CELLS)
                  if any(m in s for m in PROBE_MARKERS)), None)

print(f"{len(CODE_CELLS)} code cells; setup range = 0..{SETUP_END}"
      if SETUP_END is not None else
      f"{len(CODE_CELLS)} code cells; probe cell NOT found -- set SETUP_END by hand")

for n, (i, s) in enumerate(CODE_CELLS):
    head = next((l for l in s.splitlines() if l.strip()), "")
    flag = " [magics]" if any(l.lstrip().startswith(("!", "%"))
                              for l in s.splitlines()) else ""
    mark = "  <-- setup end" if n == SETUP_END else ""
    print(f"  [{n:2d}] nb#{i:2d} {len(s):>6}b  {head[:70]}{flag}{mark}")

print("\n--- full source of the setup cells ---")
for n, (i, s) in enumerate(CODE_CELLS[: (SETUP_END or 0) + 1]):
    print(f"\n########## CODE CELL {n} (notebook index {i}) ##########")
    print(s)

In [ ]:
CONFIRM = True
SKIP = set()          # e.g. {0} to skip a pip cell already run

# Fix: If SETUP_END was not found in the previous cell, set it manually.
# Based on the output of cell 2, 'nb# 7: Verify recurrent operating point'
# (which is CODE_CELLS index 6) seems to be the last setup cell.
if 'SETUP_END' not in globals() or SETUP_END is None:
    print("WARNING: SETUP_END not found automatically. Setting to 6 based on notebook structure.")
    SETUP_END = 6

if not CONFIRM:
    print(f"set CONFIRM = True to run cells 0..{SETUP_END}")
else:
    ip = get_ipython()
    for n, (i, s) in enumerate(CODE_CELLS[:SETUP_END + 1]):
        if n in SKIP:
            print(f"--- skipping cell {n}"); continue
        print(f"\n########## running cell {n} (notebook index {i}) ##########")
        res = ip.run_cell(s)
        if res.error_before_exec or res.error_in_exec:
            raise RuntimeError(
                f"cell {n} failed: {res.error_before_exec or res.error_in_exec}\n"
                "Fix it in place (paths may have changed) and re-run from here.")

    # the recovered cells reassign RESULT_DIR to wherever they originally
    # pointed -- put it back on Drive so nothing writes to ephemeral storage
    RESULT_DIR = DRIVE_ROOT
    print("\nRESULT_DIR reset to:", RESULT_DIR)

    # CFG shim: the older notebook version lacks a few attributes this one uses
    _defaults = dict(perm_seed=7, freeze_norm_stats=True, clean_threshold=0.95,
                     viability_threshold=0.95, operating_load=3,
                     retrieval_train_loads=(2, 3), human_loads=(3, 4, 5, 6),
                     attn_dim=256, grad_clip=1.0, lr_steer=1e-4,
                     batch_size=48, num_workers=2, max_seq_len=20)
    for k, v in _defaults.items():
        if not hasattr(CFG, k):
            setattr(CFG, k, v); print(f"  CFG.{k} defaulted to {v!r}")

    REQUIRED = ["CFG", "DEVICE", "official_model", "train_full", "train_idx",
                "test_loaders", "indices_for_loads", "move_batch",
                "response_indices", "response_mask_general", "infinite_loader",
                "make_train_loader", "freeze_norm_", "official_hidden_states",
                "HIDDEN_SIZE", "N_CLASSES", "CHANCE",
                "Visual_Serial_Recall_Recognition_Dataset"]
    lack = [k for k in REQUIRED if k not in globals()]
    print(f"\ndefined: {len(REQUIRED)-len(lack)}/{len(REQUIRED)}")
    if lack:
        print("MISSING:", lack)
        print("Extend SETUP_END, or define these by hand before continuing.")
    else:
        print("all required names present")

In [ ]:
# Patch: supply the names the old setup notebook lacks
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset

def freeze_norm_(module):
    for m in module.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d,
                          nn.SyncBatchNorm, nn.InstanceNorm1d,
                          nn.InstanceNorm2d, nn.InstanceNorm3d)):
            m.eval()

def make_train_loader(seed):
    g = torch.Generator(); g.manual_seed(int(seed))
    return DataLoader(Subset(train_full, train_idx), batch_size=CFG.batch_size,
                      shuffle=True, drop_last=True, num_workers=CFG.num_workers,
                      pin_memory=torch.cuda.is_available(), generator=g)

TASK_NAME = "VSR_Task"

def _unwrap_rnn_output(o):
    if isinstance(o, tuple): o = o[0]
    if isinstance(o, nn.utils.rnn.PackedSequence):
        o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True)
    return o

class _GRUCapture:
    def __init__(self, gru_module): self.mod, self.h, self.handle = gru_module, None, None
    def __enter__(self):
        def hook(m, i, o): self.h = _unwrap_rnn_output(o)
        self.handle = self.mod.register_forward_hook(hook); return self
    def __exit__(self, *e):
        self.handle.remove(); self.handle = None; return False

def _last_linear_out_features(module):
    last = None
    for m in module.modules():
        if isinstance(m, nn.Linear): last = m
    if last is None: raise RuntimeError("no nn.Linear in the VSR head")
    return last.out_features

_b = next(iter(test_loaders[CFG.operating_load]))
_stim, _resp, _seq_len, _ll = move_batch(_b)
official_model.eval()
with torch.no_grad(), _GRUCapture(official_model.MEM) as cap:
    _out = official_model(_stim, TASK_NAME, _seq_len)
    _logits = _out[0] if isinstance(_out, tuple) else _out
    _h = cap.h

B_probe, T_probe = _stim.shape[0], _stim.shape[1]
GRU_BATCH_FIRST = (_h.shape[0] == B_probe and _h.shape[1] <= T_probe)
if not GRU_BATCH_FIRST:
    assert _h.shape[1] == B_probe, f"unexpected GRU output {tuple(_h.shape)}"

HIDDEN_SIZE = int(_h.shape[-1])
N_CLASSES = int(_last_linear_out_features(official_model.vsr_task_head))
CHANCE = 1.0 / N_CLASSES

print("official out :", tuple(_logits.shape))
print("GRU states   :", tuple(_h.shape), "| batch_first =", GRU_BATCH_FIRST)
print("HIDDEN_SIZE  :", HIDDEN_SIZE, "| N_CLASSES:", N_CLASSES,
      "| CHANCE:", round(CHANCE, 4), "| max label:", int(_resp.max()))

assert _logits.shape[:2] == (B_probe, T_probe)
assert _logits.shape[-1] == N_CLASSES
assert int(_resp.max()) < N_CLASSES
_assumed = getattr(CFG, "n_classes_fallback", getattr(CFG, "n_classes", None))
if _assumed is not None and N_CLASSES != _assumed:
    print(f"NOTE: N_CLASSES={N_CLASSES}, not the assumed {_assumed}.")

def official_hidden_states(stim, seq_len, model=None):
    model = official_model if model is None else model
    with _GRUCapture(model.MEM) as cap:
        out = model(stim, TASK_NAME, seq_len)
    logits = out[0] if isinstance(out, tuple) else out
    h = cap.h
    if not GRU_BATCH_FIRST: h = h.transpose(0, 1)
    T = stim.shape[1]
    if h.shape[1] < T:
        h = torch.cat([h, torch.zeros(h.shape[0], T - h.shape[1], h.shape[2],
                                      dtype=h.dtype, device=h.device)], dim=1)
    return logits, h

del _b, _stim, _resp, _seq_len, _ll, _out, _logits, _h

REQUIRED = ["CFG","DEVICE","official_model","train_full","train_idx","test_loaders",
            "indices_for_loads","move_batch","response_indices","response_mask_general",
            "infinite_loader","make_train_loader","freeze_norm_","official_hidden_states",
            "HIDDEN_SIZE","N_CLASSES","CHANCE","Visual_Serial_Recall_Recognition_Dataset"]
lack = [k for k in REQUIRED if k not in globals()]
print("\ndefined:", len(REQUIRED) - len(lack), "/", len(REQUIRED))
print("MISSING:", lack or "none — continue with cell 4")

In [ ]:
# 4) Gate 1 re-check -- the official model must reproduce its own numbers
assert "official_model" in globals(), "run cell 3 with CONFIRM = True first"
official_model.eval()
for L in (2, 3, 4):
    if L in test_loaders:
        correct = total = 0
        with torch.no_grad():
            for batch in test_loaders[L]:
                stim, resp, seq_len, list_length = move_batch(batch)
                logits, _ = official_hidden_states(stim, seq_len, model=official_model)
                B = stim.shape[0]
                idx = response_indices(seq_len, list_length)
                bidx = torch.arange(B, device=DEVICE)[:, None]
                correct += (logits[bidx, idx].argmax(-1) == resp[bidx, idx]).sum().item()
                total += idx.numel()
        print(f"official recurrent load {L}: {correct/total:.4f}")

print("\nexpected from the original run (200-trial split):")
print("  L2 0.9900   L3 0.9700   L4 0.9387")
print("A mismatch means the checkpoint or repo differs from the one that")
print("produced the saved CSVs, and those CSVs can no longer be continued.")

In [ ]:
# 2) Redefine the retrieval route, hybrid, training and metrics
#
# Identical to the definitions used to produce the saved CSVs. Seeds match, so
# the regenerated models correspond to the saved ones up to kernel
# nondeterminism.

class RetrievalRoute(nn.Module):
    def __init__(self, hidden, attn_dim, n_classes, max_T,
                 n_heads=1, use_pos=False, mlp_layers=1):
        super().__init__()
        assert attn_dim % n_heads == 0
        self.n_heads, self.attn_dim = n_heads, attn_dim
        self.q_proj = nn.Linear(hidden, attn_dim, bias=False)
        self.k_proj = nn.Linear(hidden, attn_dim, bias=False)
        self.v_proj = nn.Linear(hidden, attn_dim, bias=False)

        self.use_pos = use_pos
        if use_pos:
            self.pos_q = nn.Embedding(max_T, attn_dim)
            self.pos_k = nn.Embedding(max_T, attn_dim)
            nn.init.normal_(self.pos_q.weight, std=0.02)
            nn.init.normal_(self.pos_k.weight, std=0.02)

        layers = []
        for _ in range(mlp_layers):
            layers += [nn.Linear(attn_dim, attn_dim), nn.GELU()]
        self.context_mlp = nn.Sequential(*layers)
        self.context_head = nn.Linear(attn_dim, n_classes)
        nn.init.zeros_(self.context_head.weight)
        nn.init.zeros_(self.context_head.bias)

    def readout(self, c):
        return self.context_head(self.context_mlp(c))

    def forward(self, h, seq_len):
        B, T, _ = h.shape
        nh, A = self.n_heads, self.attn_dim
        dh = A // nh
        t_idx = torch.arange(T, device=h.device)

        q, k, v = self.q_proj(h), self.k_proj(h), self.v_proj(h)
        if self.use_pos:
            q = q + self.pos_q(t_idx)[None]
            k = k + self.pos_k(t_idx)[None]

        split = lambda x: x.view(B, T, nh, dh).transpose(1, 2)
        qh, kh, vh = split(q), split(k), split(v)

        scores = torch.matmul(qh, kh.transpose(-1, -2)) / math.sqrt(dh)
        causal = torch.ones(T, T, dtype=torch.bool, device=h.device).tril(-1)
        valid_key = (t_idx[None, :] < seq_len[:, None])
        allowed = (causal[None, :, :] & valid_key[:, None, :])[:, None]

        scores = scores.masked_fill(~allowed, torch.finfo(scores.dtype).min)
        attn = scores.softmax(dim=-1)
        attn = attn * allowed.any(dim=-1, keepdim=True).to(attn.dtype)

        c = torch.matmul(attn, vh).transpose(1, 2).reshape(B, T, A)
        return c, self.readout(c), attn.mean(1)


class HybridV2(nn.Module):
    def __init__(self, official, beta=1.0, **rcfg):
        super().__init__()
        self.backbone = copy.deepcopy(official)
        self.register_buffer("beta", torch.tensor(float(beta)))
        self.retrieval = RetrievalRoute(
            HIDDEN_SIZE, rcfg.pop("attn_dim", CFG.attn_dim),
            N_CLASSES, CFG.max_seq_len, **rcfg)

    def set_beta(self, v):
        with torch.no_grad():
            self.beta.fill_(float(v))
        return self

    @property
    def beta_value(self):
        return float(self.beta.item())

    def retrieval_readout(self, c):
        return self.retrieval.readout(c)

    def retrieval_from_h(self, h, seq_len):
        c, raw, attn = self.retrieval(h, seq_len)
        return c, self.beta * raw, raw, attn

    def forward(self, stim, seq_len):
        logits_h, h = official_hidden_states(stim, seq_len, model=self.backbone)
        c, lc, raw, attn = self.retrieval_from_h(h, seq_len)
        return {"h": h, "c": c, "logits_h": logits_h, "logits_c": lc,
                "logits_c_raw": raw, "logits": logits_h + lc, "attn": attn}


def build_hybrid_v2(seed, beta=1.0, **rcfg):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return HybridV2(official_model, beta=beta, **rcfg).to(DEVICE).eval()


def _derangement(B, generator, device):
    if B < 2:
        return torch.arange(B, device=device)
    p = torch.randperm(B, generator=generator)
    for i in (p == torch.arange(B)).nonzero(as_tuple=True)[0].tolist():
        j = (i + 1) % B
        p[i], p[j] = p[j].clone(), p[i].clone()
    return p.to(device)


@torch.no_grad()
def route_accuracy(model, load, route="full", loaders=None):
    loaders = eval_loaders if loaders is None else loaders
    key = {"full": "logits", "recurrent": "logits_h",
           "retrieval": "logits_c", "retrieval_raw": "logits_c_raw"}[route]
    was = model.training; model.eval()
    correct = total = 0
    try:
        for batch in loaders[load]:
            stim, resp, seq_len, list_length = move_batch(batch)
            out = model(stim, seq_len)
            B = stim.shape[0]
            idx = response_indices(seq_len, list_length)
            bidx = torch.arange(B, device=DEVICE)[:, None]
            correct += (out[key][bidx, idx].argmax(-1) == resp[bidx, idx]).sum().item()
            total += idx.numel()
    finally:
        if was: model.train()
    return correct / total


@torch.no_grad()
def per_trial_correct(model, loader, route="retrieval_raw"):
    key = {"full": "logits", "recurrent": "logits_h",
           "retrieval": "logits_c", "retrieval_raw": "logits_c_raw"}[route]
    model.eval()
    chunks = []
    for batch in loader:
        stim, resp, seq_len, list_length = move_batch(batch)
        out = model(stim, seq_len)
        B = stim.shape[0]
        idx = response_indices(seq_len, list_length)
        bidx = torch.arange(B, device=DEVICE)[:, None]
        chunks.append((out[key][bidx, idx].argmax(-1) == resp[bidx, idx]).cpu().numpy())
    return np.concatenate(chunks, 0)


@torch.no_grad()
def official_per_trial_correct(loader):
    official_model.eval()
    chunks = []
    for batch in loader:
        stim, resp, seq_len, list_length = move_batch(batch)
        logits, _ = official_hidden_states(stim, seq_len, model=official_model)
        B = stim.shape[0]
        idx = response_indices(seq_len, list_length)
        bidx = torch.arange(B, device=DEVICE)[:, None]
        chunks.append((logits[bidx, idx].argmax(-1) == resp[bidx, idx]).cpu().numpy())
    return np.concatenate(chunks, 0)


def acc_ci(corr, n_boot=4000, seed=0):
    corr = np.asarray(corr, dtype=float)
    n = corr.shape[0]
    rng = np.random.default_rng(seed)
    boots = np.array([corr[rng.integers(0, n, n)].mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(corr.mean()), float(lo), float(hi), n


def verdict(point, lo, hi, thr=0.95):
    if lo >= thr: return "PASS (CI entirely above screen)"
    if hi < thr:  return "FAIL (CI entirely below screen)"
    return "AMBIGUOUS (CI straddles screen)"


@torch.no_grad()
def causal_metrics_v2(model, load, loaders=None, perm_seed=None):
    loaders = eval_loaders if loaders is None else loaders
    was = model.training; model.eval()
    g = torch.Generator(); g.manual_seed(CFG.perm_seed if perm_seed is None else int(perm_seed))
    clean = hp = cp = deleted = retr_only = total = 0
    sh_sum = sc_sum = 0.0
    try:
        for batch in loaders[load]:
            stim, resp, seq_len, list_length = move_batch(batch)
            out = model(stim, seq_len)
            B = stim.shape[0]
            idx = response_indices(seq_len, list_length)
            bidx = torch.arange(B, device=DEVICE)[:, None]

            h  = out["h"][bidx, idx];  c  = out["c"][bidx, idx]
            lh = out["logits_h"][bidx, idx]; lc = out["logits_c"][bidx, idx]
            y  = resp[bidx, idx]
            pi = _derangement(B, g, DEVICE)

            clean += ((lh + lc).argmax(-1) == y).sum().item()
            hp += ((model.backbone.vsr_task_head(h[pi]) + lc).argmax(-1) == y).sum().item()
            cp += ((lh + model.beta * model.retrieval_readout(c[pi])).argmax(-1) == y).sum().item()
            deleted += (lh.argmax(-1) == y).sum().item()
            retr_only += (out["logits_c_raw"][bidx, idx].argmax(-1) == y).sum().item()
            total += y.numel()
            sh_sum += lh.std(dim=-1).sum().item()
            sc_sum += lc.std(dim=-1).sum().item()
    finally:
        if was: model.train()

    a = clean / total
    denom = max(a - CHANCE, 1e-9)
    return {"accuracy": a,
            "Dh": float(np.clip((a - hp / total) / denom, 0, 1)),
            "Dc": float(np.clip((a - cp / total) / denom, 0, 1)),
            "acc_retrieval_deleted": deleted / total,
            "deletion_cost": a - deleted / total,
            "recurrent_only": deleted / total,
            "retrieval_only": retr_only / total,
            "retrieval_share": float(sc_sum / (sh_sum + sc_sum + 1e-9))}


@torch.no_grad()
def attention_map(model, load, max_batches=8, loaders=None):
    loaders = eval_loaders if loaders is None else loaders
    model.eval()
    acc = None; n = 0
    for bi, batch in enumerate(loaders[load]):
        if bi >= max_batches: break
        stim, resp, seq_len, list_length = move_batch(batch)
        out = model(stim, seq_len)
        B = stim.shape[0]
        idx = response_indices(seq_len, list_length)
        bidx = torch.arange(B, device=DEVICE)[:, None]
        a = out["attn"][bidx, idx]
        acc = a.sum(0) if acc is None else acc + a.sum(0)
        n += B
    return (acc / n).cpu().numpy()

print("definitions restored")

In [ ]:
# 6) Rebuild the evaluation set (deterministic, seed 54321, as before)
_from_setup = ["Visual_Serial_Recall_Recognition_Dataset", "CFG", "DEVICE",
               "indices_for_loads", "move_batch", "response_indices",
               "official_hidden_states", "official_model"]
_lack = [n for n in _from_setup if n not in globals()]
if _lack:
    raise NameError("missing from the setup cells: " + ", ".join(_lack)
                    + "\nRun cell 3 with CONFIRM = True first.")

if "RECURRENT_HIGH_N" not in globals() or RECURRENT_HIGH_N is None:
    _p = RESULT_DIR / "recurrent_viability_high_n.csv"
    RECURRENT_HIGH_N = pd.read_csv(_p) if _p.exists() else None
    print("RECURRENT_HIGH_N:",
          "loaded from Drive" if RECURRENT_HIGH_N is not None else "NOT FOUND")

EVAL_DIR = Path("/content/worm_vsr_eval_large")
EVAL_SAMPLES = 16000
LOADS = sorted(set(CFG.human_loads) | {2})

if "eval_full" in globals() and "eval_loaders" in globals() \
        and set(eval_loaders) >= set(LOADS):
    print("eval set already in memory, skipping regeneration")
else:
    if EVAL_DIR.exists() and any(EVAL_DIR.iterdir()):
        print(f"WARNING: {EVAL_DIR} is non-empty from a previous partial run.")
        print(f"If generation errors or counts look wrong:  !rm -rf {EVAL_DIR}")
    EVAL_DIR.mkdir(parents=True, exist_ok=True)

    random.seed(54321); np.random.seed(54321); torch.manual_seed(54321)
    t0 = time.time()
    eval_full = Visual_Serial_Recall_Recognition_Dataset(
        data_path=str(EVAL_DIR), max_seq_len=CFG.max_seq_len,
        probe_variant="Recall", grid_size=CFG.grid_size,
        list_length_options=[2, 3, 4, 5, 6, 7, 8, 9],
        distractor_difference_options=[], gen_random_trials=True,
        held_out_list_lengths=[], held_out_distractor_diffs=[],
        num_samples=EVAL_SAMPLES, img_size=CFG.img_size,
        rs_img_size=CFG.rs_img_size, write=True, split="train")
    print(f"generated {len(eval_full)} trials in {time.time()-t0:.0f}s")

    eval_loaders = {}
    for L in LOADS:
        idx = indices_for_loads(eval_full, [L])
        eval_loaders[L] = DataLoader(
            Subset(eval_full, idx), batch_size=CFG.batch_size, shuffle=False,
            drop_last=False, num_workers=CFG.num_workers,
            pin_memory=torch.cuda.is_available())
        print(f"  eval load {L}: {len(idx)} trials")

_counts = {L: len(eval_loaders[L].dataset) for L in LOADS}
assert min(_counts.values()) >= 1000, f"eval set too small: {_counts}"

chk = []
for L in LOADS:
    p, lo, hi, n = acc_ci(official_per_trial_correct(eval_loaders[L]), seed=L)
    saved = np.nan
    if RECURRENT_HIGH_N is not None:
        row = RECURRENT_HIGH_N.loc[RECURRENT_HIGH_N.load == L, "recurrent"]
        if len(row):
            saved = float(row.iloc[0])
    chk.append({"load": L, "n_trials": n, "restored": p, "ci_low": lo,
                "ci_high": hi, "saved": saved, "delta": p - saved})
    print(f"L{L}: {p:.4f} [{lo:.4f}, {hi:.4f}]  saved={saved:.4f}  delta={p-saved:+.4f}"
          if not np.isnan(saved) else f"L{L}: {p:.4f} [{lo:.4f}, {hi:.4f}]")

CHK = pd.DataFrame(chk)
display(CHK.round(5))
CHK.to_csv(RESULT_DIR / "eval_set_fidelity_check.csv", index=False)

if CHK.delta.notna().any():
    worst = CHK.delta.abs().max()
    print(f"\nlargest |delta| = {worst:.4f}")
    print("CONTINUATION: the eval set regenerated faithfully." if worst <= 0.005
          else "FRESH RUN: rebuild every table from the restored numbers and do\n"
               "not mix them with the saved CSVs in one table.")
else:
    print("\nno saved table to compare against -- fresh run by default.")

In [ ]:
# 4) Stage-2 models: load from Drive, else retrain and SAVE
STAGE2_CFG   = dict(n_heads=8, use_pos=True, mlp_layers=1, attn_dim=256)
STAGE2_TRAIN = dict(steps=30000, lr=1e-3, bs=256, loads=(2, 3))
RETRIEVAL_SEEDS = (0, 1, 2)
N_SHARDS, SHARD_SAMPLES = 5, 16000

def ckpt_path(seed):
    return CKPT_DIR / f"stage2_bigdata_seed{seed}.pt"

def save_retrieval(model, seed, extra=None):
    torch.save({"retrieval": model.retrieval.state_dict(),
                "beta": model.beta_value,
                "cfg": STAGE2_CFG, "train": STAGE2_TRAIN,
                "extra": extra or {}}, ckpt_path(seed))
    print("  saved", ckpt_path(seed).name)

def load_retrieval(seed):
    blob = torch.load(ckpt_path(seed), map_location=DEVICE, weights_only=False)
    m = build_hybrid_v2(7000 + seed, beta=blob.get("beta", 1.0), **blob["cfg"])
    m.retrieval.load_state_dict(blob["retrieval"])
    return m.to(DEVICE).eval()

need_train = [s for s in RETRIEVAL_SEEDS if not ckpt_path(s).exists()]
print("checkpoints present:",
      [s for s in RETRIEVAL_SEEDS if ckpt_path(s).exists()] or "none")
print("need training:", need_train or "none")

if need_train:
    # --- feature cache: load from Drive, else rebuild from shards and save ---
    CACHE_FILE = CACHE_DIR / "feature_cache_big.pt"
    if CACHE_FILE.exists():
        print("\nloading cached features from Drive (~1 GB, give it a minute)")
        blob = torch.load(CACHE_FILE, map_location="cpu", weights_only=False)
        FEATURE_CACHE = tuple(blob[k] for k in ("H", "R", "M", "S"))
    else:
        print("\nrebuilding feature cache from shards")

        @torch.no_grad()
        def encode_shard(dataset, loads):
            idx = indices_for_loads(dataset, loads)
            loader = DataLoader(Subset(dataset, idx), batch_size=CFG.batch_size,
                                shuffle=False, drop_last=False,
                                num_workers=CFG.num_workers,
                                pin_memory=torch.cuda.is_available())
            H, R, M, S = [], [], [], []
            official_model.eval()
            for batch in loader:
                stim, resp, seq_len, list_length = move_batch(batch)
                _, h = official_hidden_states(stim, seq_len, model=official_model)
                M.append(response_mask_general(seq_len, list_length, resp.shape[1]).cpu())
                H.append(h.half().cpu()); R.append(resp.cpu()); S.append(seq_len.cpu())
            return torch.cat(H), torch.cat(R), torch.cat(M), torch.cat(S)

        SHARD_ROOT = Path("/content/worm_vsr_shards")
        parts = []
        for k in range(N_SHARDS):
            sdir = SHARD_ROOT / f"shard{k}"; sdir.mkdir(parents=True, exist_ok=True)
            seed = 900000 + k
            random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
            ds = Visual_Serial_Recall_Recognition_Dataset(
                data_path=str(sdir), max_seq_len=CFG.max_seq_len,
                probe_variant="Recall", grid_size=CFG.grid_size,
                list_length_options=[2, 3, 4, 5, 6, 7, 8, 9],
                distractor_difference_options=[], gen_random_trials=True,
                held_out_list_lengths=[], held_out_distractor_diffs=[],
                num_samples=SHARD_SAMPLES, img_size=CFG.img_size,
                rs_img_size=CFG.rs_img_size, write=True, split="train")
            parts.append(encode_shard(ds, CFG.retrieval_train_loads))
            print(f"  shard {k}: +{parts[-1][0].shape[0]} trials")
            del ds; shutil.rmtree(sdir, ignore_errors=True)

        FEATURE_CACHE = tuple(torch.cat([p[i] for p in parts], 0) for i in range(4))
        torch.save(dict(zip(("H", "R", "M", "S"), FEATURE_CACHE)), CACHE_FILE)
        print("  cache saved to Drive:", CACHE_FILE.name)

    print("cache:", tuple(FEATURE_CACHE[0].shape),
          "| trials per load:",
          {int(L): int((FEATURE_CACHE[2].sum(1) == L).sum())
           for L in CFG.retrieval_train_loads})

    def _retrieval_params(model):
        return list(model.retrieval.parameters())

    def train_retrieval_v2(model, steps, seed, lr=1e-3, bs=256,
                           loads=(2, 3), warmup=500, log_every=10000):
        Hc, Rc, Mc, Sc = FEATURE_CACHE
        keep = torch.zeros(Hc.shape[0], dtype=torch.bool)
        for L in loads:
            keep |= (Mc.sum(1) == L)
        ki = keep.nonzero(as_tuple=True)[0]
        Hk, Rk, Mk, Sk = Hc[ki], Rc[ki], Mc[ki], Sc[ki]
        Nk = Hk.shape[0]

        for p in model.parameters():
            p.requires_grad = False
        params = _retrieval_params(model)
        for p in params:
            p.requires_grad = True

        opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: min(1.0, (s + 1) / warmup)
                           * 0.5 * (1 + math.cos(math.pi * min(1.0, s / steps))))
        g = torch.Generator(); g.manual_seed(int(seed))

        for step in range(1, steps + 1):
            model.eval(); model.retrieval.train()
            sel = torch.randint(0, Nk, (bs,), generator=g)
            h = Hk[sel].to(DEVICE).float()
            resp = Rk[sel].to(DEVICE); mask = Mk[sel].to(DEVICE)
            seq_len = Sk[sel].to(DEVICE)
            _, _, raw, _ = model.retrieval_from_h(h, seq_len)
            loss = F.cross_entropy(raw[mask], resp[mask])
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(params, CFG.grad_clip)
            opt.step(); sched.step()
            if step % log_every == 0:
                print(f"    step {step}/{steps} loss={loss.item():.4f}")
        model.eval()

    for s in need_train:
        print(f"\ntraining seed {s}")
        m = build_hybrid_v2(7000 + s, beta=1.0, **STAGE2_CFG)
        rec_before = route_accuracy(m, CFG.operating_load, "recurrent")
        train_retrieval_v2(m, seed=7000 + s, **STAGE2_TRAIN)
        assert abs(route_accuracy(m, CFG.operating_load, "recurrent") - rec_before) < 1e-9
        save_retrieval(m, s)
        del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()

stage2_big = {s: load_retrieval(s) for s in RETRIEVAL_SEEDS}
print("\nstage2_big restored:", list(stage2_big))

In [ ]:
# 5) Re-derive the two-sided screen from the restored models, and compare
rows = []
for L in sorted(eval_loaders.keys()):
    pooled = np.concatenate([per_trial_correct(stage2_big[s], eval_loaders[L],
                                               "retrieval_raw")
                             for s in RETRIEVAL_SEEDS], 0)
    p, lo, hi, n = acc_ci(pooled, seed=3000 + L)
    rec = float(RECURRENT_HIGH_N.loc[RECURRENT_HIGH_N.load == L, "recurrent"].iloc[0])
    prev = (float(POOLED_BIG.loc[POOLED_BIG.load == L, "retrieval_big_data"].iloc[0])
            if POOLED_BIG is not None else np.nan)
    rows.append({"load": L, "retrieval_restored": p, "ci_low": lo, "ci_high": hi,
                 "retrieval_saved": prev, "delta": p - prev, "recurrent": rec,
                 "verdict": verdict(p, lo, hi, CFG.viability_threshold),
                 "both_viable": (p >= CFG.viability_threshold)
                                and (rec >= CFG.viability_threshold)})

POOLED_RESTORED = pd.DataFrame(rows)
POOLED_RESTORED.to_csv(RESULT_DIR / "two_sided_screen_restored.csv", index=False)
display(POOLED_RESTORED.round(4))

print("\nViable on BOTH routes:",
      POOLED_RESTORED[POOLED_RESTORED.both_viable].load.tolist() or "NONE")
print("If |delta| is a few thousandths, the retrain reproduced. Use the")
print("RESTORED column from here on and cite the saved run as a replication.")

In [ ]:
# 6) Gate 3, cell A: corrected common-mode decomposition (retrieval route ALONE)
@torch.no_grad()
def retrieval_decomposition(model, load):
    model.eval()
    tot, n = None, 0
    for batch in eval_loaders[load]:
        stim, resp, seq_len, list_length = move_batch(batch)
        out = model(stim, seq_len)
        B = stim.shape[0]
        idx = response_indices(seq_len, list_length)
        bidx = torch.arange(B, device=DEVICE)[:, None]
        c = out["c"][bidx, idx]
        tot = c.sum(0) if tot is None else tot + c.sum(0)
        n += B
    c_bar = tot / n

    keys = ["clean", "mean_only", "dev_only"]
    correct = {k: 0 for k in keys}; total = 0
    for batch in eval_loaders[load]:
        stim, resp, seq_len, list_length = move_batch(batch)
        out = model(stim, seq_len)
        B = stim.shape[0]
        idx = response_indices(seq_len, list_length)
        bidx = torch.arange(B, device=DEVICE)[:, None]
        c = out["c"][bidx, idx]; y = resp[bidx, idx]
        for k, cv in {"clean": c,
                      "mean_only": c_bar[None].expand_as(c),
                      "dev_only": c - c_bar[None]}.items():
            correct[k] += (model.retrieval_readout(cv).argmax(-1) == y).sum().item()
        total += y.numel()
    return {"load": load, "chance": CHANCE,
            **{f"retr_{k}": correct[k] / total for k in keys}}

RDEC = pd.DataFrame([retrieval_decomposition(stage2_big[s], L)
                     for s in RETRIEVAL_SEEDS for L in (2, 3)])
RDEC = RDEC.groupby("load").mean(numeric_only=True).reset_index()
RDEC.to_csv(RESULT_DIR / "retrieval_decomposition_corrected.csv", index=False)
display(RDEC.round(4))
print("\nretr_mean_only ~ chance      -> genuinely trial-specific")
print("retr_mean_only well > chance -> part of the accuracy is a constant bias")

In [ ]:
# 7) Gate 3, cell B: the dissociation sweep -- Dc vs deletion cost across beta
#
# This is the section's figure. Pooled over the three seeds.
BETA_SWEEP = (0.125, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0)
rows = []
for b in BETA_SWEEP:
    per_seed = []
    for s in RETRIEVAL_SEEDS:
        stage2_big[s].set_beta(b)
        per_seed.append(causal_metrics_v2(stage2_big[s], CFG.operating_load))
    agg = {k: float(np.mean([d[k] for d in per_seed])) for k in per_seed[0]}
    sd = {f"{k}_sd": float(np.std([d[k] for d in per_seed])) for k in per_seed[0]}
    rows.append({"beta": b, **agg, **sd})
    print(f"beta={b:<6g} acc={agg['accuracy']:.4f} Dh={agg['Dh']:.3f} "
          f"Dc={agg['Dc']:.3f} del_acc={agg['acc_retrieval_deleted']:.4f} "
          f"del_cost={agg['deletion_cost']:+.4f} share={agg['retrieval_share']:.3f}")
for s in RETRIEVAL_SEEDS:
    stage2_big[s].set_beta(1.0)

DISSOC = pd.DataFrame(rows)
DISSOC.to_csv(RESULT_DIR / "dc_vs_deletion_dissociation.csv", index=False)
display(DISSOC[["beta", "accuracy", "Dh", "Dc",
                "acc_retrieval_deleted", "deletion_cost",
                "retrieval_share"]].round(4))

ok = DISSOC[(DISSOC.Dc > DISSOC.Dh) & (DISSOC.deletion_cost > 0.05)]
print("\nbeta values giving retrieval dominance that survives deletion:",
      ok.beta.tolist() or "NONE")
if not len(ok):
    print("=> Gate 3 cannot be passed by beta calibration. The protocol's")
    print("   'construct a retrieval-dominant hybrid, require Dc > Dh' step is")
    print("   unreachable with the backbone frozen. Proceed to cell 8.")

In [ ]:
# 8) Gate 3, cell C: warm-start pilot, with checkpointing every log interval
#
# Declared criterion: deletion cost >= 0.50 at L3 with clean accuracy >= 0.95.
# A middle result (cost rising to ~0.15) counts as NOT viable.

WARM_STEPS, WARM_LOG = 4000, 500
WARM_CKPT = CKPT_DIR / "warmstart_pilot.pt"
WARM_HIST_CSV = RESULT_DIR / "warmstart_free_hybrid.csv"

def train_joint_v2(model, steps, lam, seed, log_every, load, start_step=0, hist=None):
    for p in model.parameters():
        p.requires_grad = True
    torch.manual_seed(int(seed))
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(int(seed))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr_steer, weight_decay=0.0)
    it = infinite_loader(make_train_loader(seed))
    hist = hist if hist is not None else []

    for step in range(start_step + 1, steps + 1):
        model.train()
        if CFG.freeze_norm_stats: freeze_norm_(model)
        stim, resp, seq_len, list_length = move_batch(next(it))
        out = model(stim, seq_len)
        mask = response_mask_general(seq_len, list_length, resp.shape[1])
        loss = F.cross_entropy(out["logits"][mask], resp[mask])
        if lam > 0:
            loss = loss + lam * out["logits_c"][mask].norm(dim=-1).mean()
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        opt.step()

        if step == 1 or step % log_every == 0:
            met = causal_metrics_v2(model, load)
            hist.append({"step": step, "loss": float(loss.item()), **met})
            print(f"  step {step:5d} acc={met['accuracy']:.4f} Dh={met['Dh']:.3f} "
                  f"Dc={met['Dc']:.3f} del_cost={met['deletion_cost']:+.4f}")
            # checkpoint so a disconnect costs one interval, not the run
            torch.save({"model": model.state_dict(), "step": step, "hist": hist},
                       WARM_CKPT)
            pd.DataFrame(hist).to_csv(WARM_HIST_CSV, index=False)
    model.eval()
    return pd.DataFrame(hist)

warm = copy.deepcopy(stage2_big[RETRIEVAL_SEEDS[0]]).to(DEVICE)
warm.set_beta(1.0)
start_step, hist = 0, []
if WARM_CKPT.exists():
    blob = torch.load(WARM_CKPT, map_location=DEVICE, weights_only=False)
    warm.load_state_dict(blob["model"]); start_step = blob["step"]; hist = blob["hist"]
    print(f"resuming warm start from step {start_step}")

print("before warm start:",
      {k: round(v, 4) for k, v in causal_metrics_v2(warm, CFG.operating_load).items()})
WARM_HIST = train_joint_v2(warm, WARM_STEPS, lam=0.0, seed=424242,
                           log_every=WARM_LOG, load=CFG.operating_load,
                           start_step=start_step, hist=hist)
display(WARM_HIST[["step", "accuracy", "Dh", "Dc",
                   "acc_retrieval_deleted", "deletion_cost"]].round(4))

final = causal_metrics_v2(warm, CFG.operating_load)
print("\n" + "=" * 70)
print(f"deletion cost after warm start: {final['deletion_cost']:+.4f} "
      f"(clean accuracy {final['accuracy']:.4f})")
print("paper's never arm, for reference: deletion leaves 0.078 accuracy")
print("=" * 70)
if final["deletion_cost"] >= 0.50 and final["accuracy"] >= 0.95:
    print("OPTION A VIABLE: a genuine retrieval-dominant PRE exists.")
    print("Retitle to 'warm-started from the official checkpoint' and report")
    print("the distance travelled from the official weights (cell 9).")
else:
    print("OPTION A NOT VIABLE at this budget. Report Option B: route")
    print("availability plus the contribution-necessity dissociation, and make")
    print("no handover claim in this setting.")

In [ ]:
# 9) Distance travelled from the official checkpoint, and a final save
@torch.no_grad()
def checkpoint_distance(model, reference=None):
    reference = official_model if reference is None else reference
    ref = dict(reference.named_parameters())
    rows = [{"param": n,
             "rel_change": float((p - ref[n]).norm() / (ref[n].norm() + 1e-9)),
             "n": p.numel()}
            for n, p in model.backbone.named_parameters() if n in ref]
    df = pd.DataFrame(rows)
    total = float(np.sqrt((df.rel_change ** 2 * df.n).sum() / df.n.sum()))
    return df.sort_values("rel_change", ascending=False), total

DIST_DF, DIST_TOTAL = checkpoint_distance(warm)
DIST_DF.to_csv(RESULT_DIR / "warmstart_checkpoint_distance.csv", index=False)
display(DIST_DF.head(12).round(5))
print(f"\nparameter-weighted movement from official checkpoint: {DIST_TOTAL:.4f}")
print("official recurrent L3:",
      round(float(RECURRENT_HIGH_N.loc[RECURRENT_HIGH_N.load == 3, 'recurrent'].iloc[0]), 4))
print("warm-started recurrent-only L3:",
      round(causal_metrics_v2(warm, CFG.operating_load)["recurrent_only"], 4))

print("\nfiles now in Drive:")
for f in sorted(RESULT_DIR.glob("*.csv")): print("  results/", f.name)
for f in sorted(CKPT_DIR.glob("*.pt")):    print("  checkpoints/", f.name)
for f in sorted(CACHE_DIR.glob("*.pt")):   print("  cache/", f.name)

## Reading order

**Cell 6, the delta column.** If |delta| exceeds ~0.005 the evaluation set did
not regenerate, nothing here is a continuation of the saved run, and every
table has to be rebuilt from the restored numbers.

**Cell 10, the β sweep.** If no β gives retrieval dominance that survives
deletion, then step 5 of the original protocol — construct a retrieval-dominant
hybrid, require `Dc > Dh` — is unreachable with the backbone frozen. That is a
protocol result, not a tuning failure, and it is the section's figure.

**Cell 11, the warm-start pilot.** Declared criterion: deletion cost ≥ 0.50 at
clean accuracy ≥ 0.95. A middle outcome (cost rising to ~0.15) counts as not
viable — it is nowhere near the paper's never-arm regime, where deletion leaves
0.078, and it is the most tempting result to over-read.

Viable → the four-arm experiment is meaningful; retitle to "warm-started from
the official checkpoint" and report the distance from cell 12. Not viable →
write up route availability plus the contribution–necessity dissociation, and
make no handover claim in this setting.

## Renumbering note

Cells 7–12 keep the internal comment numbering of the notebook they came from
(`# 4)` through `# 9)`), which no longer matches their position here. The code
is unchanged; only the position differs.

In [ ]:
# Option A rerun -- enlarged data, low LR, clean-accuracy floor
#
# Q.7 currently has to say "the pilot degraded the backbone" rather than
# "unfreezing does not help". This closes that gap.
#
# Three changes from the first pilot:
#   1. ~16000 training trials at loads 2-3 instead of 4000. The cached hidden
#      states cannot be reused here -- joint training updates the backbone, so
#      it needs the stimuli, and the shards were deleted after encoding.
#   2. lr 2e-5 instead of 1e-4, with 200-step warmup.
#   3. A declared clean-accuracy floor: two consecutive log points below 0.95
#      stops the run and reports DEGRADED, which is a different outcome from
#      NOT VIABLE and must not be read as one.
#
# Declared criterion, unchanged: deletion cost >= 0.50 at clean accuracy >= 0.95.

for _n in ["stage2_big", "eval_loaders", "causal_metrics_v2", "CFG", "DEVICE",
           "Visual_Serial_Recall_Recognition_Dataset", "indices_for_loads",
           "move_batch", "response_mask_general", "infinite_loader",
           "freeze_norm_", "RESULT_DIR", "CKPT_DIR"]:
    assert _n in globals(), f"missing: {_n} -- run the full notebook through cell 8 first"

WARM2_STEPS   = 6000
WARM2_LOG     = 500
WARM2_LR      = 2e-5
WARM2_WARMUP  = 200
WARM2_SEED    = 515151
ACC_FLOOR     = 0.95
WARM2_CKPT    = CKPT_DIR / "warmstart_pilot_v2.pt"
WARM2_CSV     = RESULT_DIR / "warmstart_free_hybrid_v2.csv"

# ---- enlarged training set (images kept, unlike the encode-and-delete shards)
WARM_DATA_DIR = Path("/content/worm_vsr_warm_train")
WARM_SAMPLES  = 64000          # ~8000 trials per load -> ~16000 at loads 2-3

if "warm_train_full" not in globals():
    WARM_DATA_DIR.mkdir(parents=True, exist_ok=True)
    random.seed(777777); np.random.seed(777777); torch.manual_seed(777777)
    t0 = time.time()
    warm_train_full = Visual_Serial_Recall_Recognition_Dataset(
        data_path=str(WARM_DATA_DIR), max_seq_len=CFG.max_seq_len,
        probe_variant="Recall", grid_size=CFG.grid_size,
        list_length_options=[2, 3, 4, 5, 6, 7, 8, 9],
        distractor_difference_options=[], gen_random_trials=True,
        held_out_list_lengths=[], held_out_distractor_diffs=[],
        num_samples=WARM_SAMPLES, img_size=CFG.img_size,
        rs_img_size=CFG.rs_img_size, write=True, split="train")
    print(f"generated in {time.time()-t0:.0f}s")

warm_idx = indices_for_loads(warm_train_full, CFG.retrieval_train_loads)
print("warm-start training trials:", len(warm_idx))
assert len(warm_idx) >= 12000, "enlarged set smaller than expected"
!df -h /content | tail -1

def make_warm_loader(seed):
    g = torch.Generator(); g.manual_seed(int(seed))
    return DataLoader(Subset(warm_train_full, warm_idx),
                      batch_size=CFG.batch_size, shuffle=True, drop_last=True,
                      num_workers=CFG.num_workers,
                      pin_memory=torch.cuda.is_available(), generator=g)

# ---- training loop ----------------------------------------------------------
def train_joint_v3(model, steps, seed, load, start_step=0, hist=None):
    for p in model.parameters():
        p.requires_grad = True
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    opt = torch.optim.AdamW(model.parameters(), lr=WARM2_LR, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / WARM2_WARMUP))
    it = infinite_loader(make_warm_loader(seed))
    hist = hist if hist is not None else []
    below = 0
    status = "COMPLETED"

    for step in range(start_step + 1, steps + 1):
        model.train()
        if CFG.freeze_norm_stats:
            freeze_norm_(model)

        stim, resp, seq_len, list_length = move_batch(next(it))
        out = model(stim, seq_len)
        mask = response_mask_general(seq_len, list_length, resp.shape[1])
        loss = F.cross_entropy(out["logits"][mask], resp[mask])   # unpriced

        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        opt.step(); sched.step()

        if step == 1 or step % WARM2_LOG == 0:
            met = causal_metrics_v2(model, load)
            hist.append({"step": step, "loss": float(loss.item()), **met})
            print(f"  step {step:5d} acc={met['accuracy']:.4f} "
                  f"rec_only={met['recurrent_only']:.4f} "
                  f"Dh={met['Dh']:.3f} Dc={met['Dc']:.3f} "
                  f"del_cost={met['deletion_cost']:+.4f}")
            torch.save({"model": model.state_dict(), "step": step, "hist": hist},
                       WARM2_CKPT)
            pd.DataFrame(hist).to_csv(WARM2_CSV, index=False)

            below = below + 1 if met["accuracy"] < ACC_FLOOR else 0
            if below >= 2:
                status = "DEGRADED"
                print(f"\n  stopping: clean accuracy below {ACC_FLOOR} at two "
                      "consecutive log points")
                break

    model.eval()
    return pd.DataFrame(hist), status

# ---- run --------------------------------------------------------------------
warm2 = copy.deepcopy(stage2_big[RETRIEVAL_SEEDS[0]]).to(DEVICE)
warm2.set_beta(1.0)

start_step, hist = 0, []
if WARM2_CKPT.exists():
    blob = torch.load(WARM2_CKPT, map_location=DEVICE, weights_only=False)
    warm2.load_state_dict(blob["model"]); start_step = blob["step"]; hist = blob["hist"]
    print(f"resuming from step {start_step}")

_before = causal_metrics_v2(warm2, CFG.operating_load)
print("before:", {k: round(v, 4) for k, v in _before.items()})
print(f"\nunpriced joint training, lr={WARM2_LR}, {len(warm_idx)} trials\n")

WARM2_HIST, STATUS = train_joint_v3(warm2, WARM2_STEPS, WARM2_SEED,
                                    CFG.operating_load, start_step, hist)
display(WARM2_HIST[["step", "accuracy", "recurrent_only", "Dh", "Dc",
                    "deletion_cost"]].round(4))

final = causal_metrics_v2(warm2, CFG.operating_load)
peak_del = float(WARM2_HIST.deletion_cost.max())

print("\n" + "=" * 70)
print(f"status                : {STATUS}")
print(f"final deletion cost   : {final['deletion_cost']:+.4f}")
print(f"peak deletion cost    : {peak_del:+.4f}")
print(f"final clean accuracy  : {final['accuracy']:.4f}")
print(f"final recurrent-only  : {final['recurrent_only']:.4f}")
print("reference: paper's never arm leaves 0.078 accuracy under deletion")
print("=" * 70)

if STATUS == "DEGRADED":
    print("DEGRADED: the backbone lost the task before the question was answered.")
    print("Not a negative result. Lower WARM2_LR further and rerun.")
elif final["deletion_cost"] >= 0.50 and final["accuracy"] >= ACC_FLOOR:
    print("OPTION A VIABLE: retrieval became load-bearing. Retitle to")
    print("'warm-started from the official checkpoint', report the parameter")
    print("distance, and run the four arms.")
else:
    print("OPTION A NOT VIABLE, and now cleanly so: the backbone held the task")
    print(f"(clean accuracy {final['accuracy']:.4f}) on {len(warm_idx)} trials and")
    print("retrieval was still never recruited. Q.7 can state this without the")
    print("'the pilot degraded the backbone' caveat.")

if "checkpoint_distance" in globals():
    _, d = checkpoint_distance(warm2)
    print(f"\nparameter-weighted movement from official checkpoint: {d:.4f}")

print(f"\nsaved: {WARM2_CSV.name}, {WARM2_CKPT.name}")
print(f"free disk when done:  !rm -rf {WARM_DATA_DIR}")

In [ ]:
H = pd.read_csv(RESULT_DIR / "warmstart_free_hybrid_v2.csv")
D = pd.read_csv(RESULT_DIR / "dc_vs_deletion_dissociation.csv")
cols = ["retrieval_share", "Dc", "accuracy", "deletion_cost"]
pts = pd.concat([H.assign(source="warm start (learned)")[cols + ["source"]],
                 D.assign(source="beta sweep (imposed)")[cols + ["source"]]])
display(pts.sort_values("retrieval_share").round(4))

import matplotlib.pyplot as plt
for s, g in pts.groupby("source"):
    plt.scatter(g.retrieval_share, g.Dc, label=s)
plt.xlabel("retrieval contribution share"); plt.ylabel("$D_c$")
plt.legend(); plt.grid(alpha=.3); plt.show()

for s, g in pts.groupby("source"):
    print(f"{s}: r(share, Dc) = {g.retrieval_share.corr(g.Dc):+.3f}  n={len(g)}")

In [ ]:
learned = H.iloc[1:].assign(source="learned")          # drop step 1 = the beta=1 anchor
imposed = D.assign(source="imposed")
from scipy.stats import spearmanr
for nm, g in [("imposed", imposed), ("learned", learned)]:
    print(f"{nm}: spearman(share, Dc) = {spearmanr(g.retrieval_share, g.Dc).statistic:+.3f}  n={len(g)}")

pred = np.interp(learned.retrieval_share, imposed.retrieval_share, imposed.Dc)
off = learned.Dc.values - pred
ok = (learned.retrieval_share >= imposed.retrieval_share.min()) & \
     (learned.retrieval_share <= imposed.retrieval_share.max())
print(f"learned minus imposed at matched share: {off[ok.values].mean():+.3f} "
      f"(sd {off[ok.values].std():.3f}, n={ok.sum()})")

allp = pd.concat([learned, imposed])
print(f"\nshare {allp.retrieval_share.min():.3f}-{allp.retrieval_share.max():.3f} | "
      f"Dc {allp.Dc.min():.3f}-{allp.Dc.max():.3f} | "
      f"accuracy {allp.accuracy.min():.4f}-{allp.accuracy.max():.4f} | "
      f"deletion cost {allp.deletion_cost.min():+.4f}-{allp.deletion_cost.max():+.4f}")

In [ ]:
DENSE_BETA = (0.15, 0.175, 0.20, 0.225, 0.275, 0.30, 0.35, 0.40, 0.45, 0.55, 0.60, 0.70)
rows = []
for b in DENSE_BETA:
    per = []
    for s in RETRIEVAL_SEEDS:
        stage2_big[s].set_beta(b)
        per.append(causal_metrics_v2(stage2_big[s], CFG.operating_load))
    rows.append({"beta": b, **{k: float(np.mean([d[k] for d in per])) for k in per[0]}})
    print(f"beta={b:<6g} share={rows[-1]['retrieval_share']:.3f} "
          f"Dc={rows[-1]['Dc']:.3f} acc={rows[-1]['accuracy']:.4f} "
          f"del={rows[-1]['deletion_cost']:+.4f}")
for s in RETRIEVAL_SEEDS:
    stage2_big[s].set_beta(1.0)

imposed2 = pd.concat([imposed, pd.DataFrame(rows).assign(source="imposed")]) \
             .sort_values("retrieval_share").reset_index(drop=True)
imposed2.to_csv(RESULT_DIR / "dc_vs_share_dense.csv", index=False)

ok = (learned.retrieval_share.between(imposed2.retrieval_share.min(),
                                      imposed2.retrieval_share.max())).values
vert = learned.Dc.values - np.interp(learned.retrieval_share, imposed2.retrieval_share, imposed2.Dc)
horiz = learned.retrieval_share.values - np.interp(learned.Dc, imposed2.Dc, imposed2.retrieval_share)
print(f"\nvertical  (Dc at matched share): {vert[ok].mean():+.3f} (sd {vert[ok].std():.3f})")
print(f"horizontal (share at matched Dc): {horiz[ok].mean():+.3f} (sd {horiz[ok].std():.3f})")
print("whichever has the smaller sd is the better description of the offset")